## Niveau Expert

source principal pour la doc : https://www.python.org/
Pour les widgets : https://ipywidgets.readthedocs.io/en/stable/
Pour les pipelines Gstreamer : https://gstreamer.freedesktop.org/documentation/
Pour les graphes aevc Matplotlib pyplot : https://matplotlib.org/stable/api/pyplot_summary.html et surtout les TP de l'an dernier en R206 

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import subprocess
import glob
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import fft, fftshift

liste_fichiers = glob.glob("*.mp3") + glob.glob("*.wav") + glob.glob("*.flac")

menu_morceau = widgets.Dropdown(options=liste_fichiers, description='Morceau :')

texte_hz = widgets.FloatText(value=44100, description='Hz :', layout=widgets.Layout(width='156px'))
plage_hz = widgets.FloatSlider(value=44100, min=8000, max=96000, step=100, orientation='vertical', description='Hz')
widgets.jslink((texte_hz, 'value'), (plage_hz, 'value'))

texte_bits = widgets.IntText(value=16, description='Bits :', layout=widgets.Layout(width='156px'))
plage_bits = widgets.IntSlider(value=16, min=8, max=32, step=8, orientation='vertical', description='Bits')
widgets.jslink((texte_bits, 'value'), (plage_bits, 'value'))

texte_vol = widgets.FloatText(value=1.0, description='Vol :', layout=widgets.Layout(width='156px'))
plage_vol = widgets.FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, orientation='vertical', description='Vol')
widgets.jslink((texte_vol, 'value'), (plage_vol, 'value'))

case_pochette = widgets.Checkbox(description='Pochette', value=True, style={'description_width': 'initial'})
case_visuel = widgets.Checkbox(description='Visuel', value=False, style={'description_width': 'initial'})
case_infos = widgets.Checkbox(description='Infos', value=False, style={'description_width': 'initial'})
case_export = widgets.Checkbox(description='Exporter', value=False, style={'description_width': 'initial'})

boite_options = widgets.HBox([case_visuel, case_pochette, case_infos, case_export])

renommage_fichier = widgets.Text(
    value='mon_export.wav', 
    description='Nom export :', 
    placeholder='Nom du fichier.wav',
    style={'description_width': 'initial'}
)

bouton_lancer = widgets.Button(description='Lancer', button_style='success', icon='play')
bouton_reptemporelle = widgets.Button(description='Afficher la représentation temporelle', button_style='info', icon='area-chart', layout=widgets.Layout(width='auto'))
bouton_spctramplitude = widgets.Button(description='Afficher le spectre d amplitude', button_style='info', icon='signal', layout=widgets.Layout(width='auto'))
bouton_quantif = widgets.Button(description='Analyse Erreur de quantification', button_style='info', icon='bar-chart', layout=widgets.Layout(width='auto'))
bouton_comparaison_quantif = widgets.Button(description='Comparaison Bits (2,4,8,12)', button_style='info', icon='th-large', layout=widgets.Layout(width='auto'))
sortie_texte = widgets.Output()


def action_lancer(b):

    fichier = menu_morceau.value

    image = fichier.replace(".mp3", ".jpg").replace(".wav", ".jpg").replace(".flac", ".jpg")
    
    with sortie_texte:
        if case_infos.value:

            res = subprocess.check_output(f'gst-discoverer-1.0 "{fichier}"', shell=True)
            print(res.decode())

        valeur_hz = int(plage_hz.value)
        valeur_bits = int(plage_bits.value)
        valeur_vol = float(plage_vol.value)
        
        filtres = ' ! audioresample ! audio/x-raw,rate=' + str(valeur_hz)
        filtres = filtres + ',format=S' + str(valeur_bits) + 'LE'
        filtres = filtres + ' ! volume volume=' + str(valeur_vol)
        
        commande_finale = ""

        if fichier.endswith(".mp3"):
            
            base = 'gst-launch-1.0 filesrc location="' + fichier + '" ! mpegaudioparse ! mpg123audiodec ! audioconvert ' + filtres
            
            if case_export.value:
                nom = renommage_fichier.value
                commande_finale = base + ' ! wavenc ! filesink location="' + nom + '"'
            else:
                if case_visuel.value:
                    commande_finale = base + ' ! tee name=t t. ! queue ! autoaudiosink t. ! queue ! wavescope ! videoconvert ! autovideosink'
                else:
                    commande_finale = base + ' ! autoaudiosink'
                
                if case_pochette.value:
                    commande_finale = commande_finale + ' filesrc location="' + image + '" ! jpegdec ! imagefreeze ! videoconvert ! autovideosink'

        elif fichier.endswith(".wav"):
            base = 'gst-launch-1.0 filesrc location="' + fichier + '" ! wavparse ! audioconvert ' + filtres
            
            if case_export.value:
                nom = renommage_fichier.value
                commande_finale = base + ' ! wavenc ! filesink location="' + nom + '"'
            else:
                if case_visuel.value:
                    commande_finale = base + ' ! tee name=t t. ! queue ! autoaudiosink t. ! queue ! wavescope ! videoconvert ! autovideosink'
                else:
                    commande_finale = base + ' ! autoaudiosink'
                
                if case_pochette.value:
                    commande_finale = commande_finale + ' filesrc location="' + image + '" ! jpegdec ! imagefreeze ! videoconvert ! autovideosink'

        elif fichier.endswith(".flac"):
            base = 'gst-launch-1.0 filesrc location="' + fichier + '" ! flacparse ! flacdec ! audioconvert ' + filtres
            
            if case_export.value:
                nom = renommage_fichier.value
                commande_finale = base + ' ! wavenc ! filesink location="' + nom + '"'
            else:
                if case_visuel.value:
                    commande_finale = base + ' ! tee name=t t. ! queue ! autoaudiosink t. ! queue ! wavescope ! videoconvert ! autovideosink'
                else:
                    commande_finale = base + ' ! autoaudiosink'
                
                if case_pochette.value:
                    commande_finale = commande_finale + ' filesrc location="' + image + '" ! jpegdec ! imagefreeze ! videoconvert ! autovideosink'

        processus_en_cours = subprocess.Popen(commande_finale, shell=True)
        
bouton_lancer.on_click(action_lancer)

fich = menu_morceau.value

def lancer_rep_temporelle(fich):
    data, samplerate = sf.read(fich)
    
    if data.ndim == 1:
        print("Fichier MONO")
        gauche = data 
        droite = data
    else:
        print("Fichier STEREO")
        gauche = data[:, 0]
        droite = data[:, 1]
        
    return gauche, droite

fich = menu_morceau.value

def spectreAmpl(b):

    with sortie_texte:
        fich = menu_morceau.value
        
        # On lit les données
        gauche, droite = lancer_rep_temporelle(fich)
        Fe = sf.info(fich).samplerate
        
        duree_gauche = len(gauche) / Fe
        tg = np.linspace(0, duree_gauche, len(gauche))
        
        duree_droite = len(droite) / Fe
        td = np.linspace(0, duree_droite, len(droite))

        # On utilise plt. (de matplotlib)
        plt.figure(figsize = (20, 5))
        
        plt.subplot(121)
        plt.plot(tg, gauche, "b-")
        plt.title(f"Signal gauche - {fich}")
        plt.grid()
        
        plt.subplot(122)
        plt.plot(td, droite, "r-")
        plt.title(f"Signal droite - {fich}")
        plt.grid()
        
        plt.show()

bouton_reptemporelle.on_click(spectreAmpl)

fich = menu_morceau.value

def plotFreq(b):

    with sortie_texte:
        fich = menu_morceau.value
        
        gauche, droite = lancer_rep_temporelle(fich)
        fe = sf.info(fich).samplerate

        longueurG = len(gauche)
        
        signal_frequentiel = fft(gauche)
        signal_centre = fftshift(signal_frequentiel)
        specOrigG = np.abs(signal_centre) / longueurG
        freqG = np.arange(-fe/2, fe/2, fe/longueurG) 
        
        plt.figure(figsize=(20, 5))
        
        plt.subplot(121)
        plt.plot(freqG, specOrigG, 'r-')
        plt.grid()
        plt.title(f'Spectre Gauche - {fich}', fontsize=16)
        plt.xlim([-fe/32, fe/32]) 
        plt.xlabel('fréquence (Hz)', fontsize=16)
        plt.ylabel('Amplitude', fontsize=16)

        longueurD = len(droite)
        
        signal_frequentiel_D = fft(droite)
        signal_centre_D = fftshift(signal_frequentiel_D)
        specOrigD = np.abs(signal_centre_D) / longueurD
        freqD = np.arange(-fe/2, fe/2, fe/longueurD)
        
        plt.subplot(122)
        plt.plot(freqD, specOrigD, 'b-')
        plt.grid()
        plt.title(f'Spectre Droit - {fich}', fontsize=16)
        plt.xlim([-fe/32, fe/32])
        plt.xlabel('fréquence (Hz)', fontsize=16)
        plt.ylabel('Amplitude', fontsize=16)
        
        plt.show() 

bouton_spctramplitude.on_click(plotFreq)

def Erreur_quantif(b):
    with sortie_texte:
        fich = menu_morceau.value

        data, samplerate = sf.read(fich)
        signal = data.flatten()

        debut = len(signal) // 2
        fin = debut + 50
        t = np.arange(0, 50)
        s = signal[debut:fin]
        
        n_bits = 3 
        
        facteur = 2 ** (n_bits - 1)
        sq = np.round(s * facteur) / facteur
        erreur = s - sq
        
        plt.figure(figsize=(14, 6))
        
        plt.plot(t, s, label="Signal original", color='blue')
        plt.step(t, sq, label="Signal Quantifiée")
        plt.plot(t, erreur, label="Erreur de quantification", color='green')
        
        plt.xlabel("NbEchantillons")
        plt.ylabel("Amplitude")
        plt.title(f"Dégradation à {n_bits} bits")
        plt.legend()
        plt.grid(True)
        plt.show()
        
bouton_quantif.on_click(Erreur_quantif)

def comparaison_quantif(b):
    with sortie_texte:
        fich = menu_morceau.value
        data, samplerate = sf.read(fich)
        signal = data.flatten()

        milieu = len(signal) // 2
        echantillons = signal[milieu : milieu + 50]

        axe_x = np.arange(0, 50)


        plt.figure(figsize=(10, 2))

        n_bits = 2
        facteur = 2 ** (n_bits - 1)
        signal_quantifie = np.round(echantillons * facteur) / facteur
        
        plt.plot(axe_x, echantillons, 'b', label="Original")
        plt.step(axe_x, signal_quantifie, 'r', label="2 bits")
        plt.title("Quantification sur 2 bits")
        plt.legend()
        plt.grid()
        plt.show()

        plt.figure(figsize=(10, 2))
        n_bits = 4
        facteur = 2 ** (n_bits - 1)
        signal_quantifie = np.round(echantillons * facteur) / facteur
        
        plt.plot(axe_x, echantillons, 'b', label="Original")
        plt.step(axe_x, signal_quantifie, 'r', label="4 bits")
        plt.title("Quantification sur 4 bits")
        plt.legend()
        plt.grid()
        plt.show()

        plt.figure(figsize=(10, 2))
        n_bits = 8
        facteur = 2 ** (n_bits - 1)
        signal_quantifie = np.round(echantillons * facteur) / facteur
        
        plt.plot(axe_x, echantillons, 'b', label="Original", alpha=0.5)
        plt.step(axe_x, signal_quantifie, 'r', label="8 bits", where='mid')
        plt.title("Quantification sur 8 bits")
        plt.legend()
        plt.grid()
        plt.show()

        plt.figure(figsize=(10, 2))
        n_bits = 12
        facteur = 2 ** (n_bits - 1)
        signal_quantifie = np.round(echantillons * facteur) / facteur
        
        plt.plot(axe_x, echantillons, 'b', label="Original", alpha=0.5)
        plt.step(axe_x, signal_quantifie, 'r', label="12 bits", where='mid')
        plt.title("Quantification sur 12 bits")
        plt.legend()
        plt.grid()
        plt.show()
        
bouton_comparaison_quantif.on_click(comparaison_quantif)

titre_box = widgets.HTML("<h2 style='text-align:center; color:#0400FF; margin:0;'>The Codec Player Studio</h2>")

box_sliders = widgets.HBox([
    widgets.VBox([texte_hz, plage_hz]),
    widgets.VBox([texte_bits, plage_bits]),
    widgets.VBox([texte_vol, plage_vol]),
], layout=widgets.Layout(justify_content='space-around', width='100%'))

interface_stylee = widgets.VBox([
    titre_box,
    menu_morceau,
    widgets.HTML("<br>"),
    box_sliders,
    widgets.HTML("<br>"),
    renommage_fichier,
    boite_options,
    widgets.HTML("<br>"),
    bouton_lancer,
    bouton_reptemporelle,
    bouton_spctramplitude,
    bouton_quantif,
    bouton_comparaison_quantif
    
])

interface_stylee.layout.border = '5px solid #0400FF'
interface_stylee.layout.border_radius = '10px'
interface_stylee.layout.padding = '20px'
interface_stylee.layout.background_color = '#f7f9f9'
interface_stylee.layout.width = '90%'
interface_stylee.layout.height = 'auto'

display(interface_stylee)
display(sortie_texte)

Output()